<a href="https://colab.research.google.com/github/Rajjprateek/AI-Workshop-May-2025/blob/Text-Generation-Project/Text_Generation_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Import dependencies
import numpy
import sys
import nltk
nltk.download('stopwords')
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
# Changed np_utils to to_categorical as np_utils is deprecated/moved
from keras.utils import to_categorical
from keras.callbacks import ModelCheckpoint

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
# Upload file
from google.colab import files
uploaded = files.upload()

# Read uploaded fil
file_content = open("frankenstein-2.txt", encoding="utf-8").read()

Saving frankenstein-2.txt to frankenstein-2.txt


In [16]:
# Tokenization and standardization
def tokenize_words(input):
    input = input.lower()
    tokenizer = RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(input)
    filtered = filter(lambda token: token not in stopwords.words('english'), tokens)
    return " ".join(filtered)
processed_inputs = tokenize_words(file_content)

In [17]:
# Changing characters to numbers
chars = sorted(list(set(processed_inputs)))
char_to_num = dict((c, i) for i, c in enumerate(chars))

In [18]:
# Check if words to chars or chars to num (?!) has worked?
input_len = len(processed_inputs)
vocab_len = len(chars)
print ("Total number of characters:", input_len)
print ("Total vocab:", vocab_len)

Total number of characters: 262517
Total vocab: 37


In [19]:
# seq length
seq_length = 100
x_data = []
y_data = []

In [20]:
# Loop through the sequnece
for i in range(0, input_len - seq_length, 1):
    in_seq = processed_inputs[i:i + seq_length]
    out_seq = processed_inputs[i + seq_length]
    x_data.append([char_to_num[char] for char in in_seq])
    y_data.append(char_to_num[out_seq])

n_patterns = len(x_data)
print ("Total Patterns:", n_patterns)

Total Patterns: 262417


In [21]:
# convert input sequence to np aaray and so on
import numpy # Import numpy
x = numpy.reshape(x_data, (n_patterns, seq_length, 1))
x = x/float(vocab_len)

In [22]:
# one-hot encoding
from tensorflow.keras.utils import to_categorical

y = to_categorical(y_data)

In [23]:
# Creating the model
model = Sequential()
model.add(LSTM(256, input_shape=(x.shape[1], x.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(256, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [24]:
# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [26]:
# Saving weights
filepath = 'model_weights_saved.keras' # Changed the file extension to .keras
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose = 1, save_best_only=True, mode='min')
desired_callbacks = [checkpoint]

In [28]:
# Fit model and let it train
model.fit(x,y, epochs=4, batch_size=256, callbacks=desired_callbacks)

Epoch 1/4
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 2.9692
Epoch 1: loss improved from inf to 2.91671, saving model to model_weights_saved.keras
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 4275s 4s/step - loss: 2.9692
Epoch 2/4
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 2.6975
Epoch 2: loss improved from 2.91671 to 2.65601, saving model to model_weights_saved.keras
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 4292s 4s/step - loss: 2.6975
Epoch 3/4
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 2.5297
Epoch 3: loss improved from 2.65601 to 2.48642, saving model to model_weights_saved.keras
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 4271s 4s/step - loss: 2.5296
Epoch 4/4
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - loss: 2.3562
Epoch 4: loss improved from 2.48642 to 2.31806, saving model to model_weights_saved.keras
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 4189s 4s/step - loss: 2.3561


In [29]:
# Recompile model with the saved weights
filename = 'model_weights_saved.keras'
model.load_weights(filename)
model.compile(loss='categorical_crossentropy', optimizer='adam')

In [30]:
# Output of the model back into characters
num_to_char = dict((i, c) for i, c in enumerate(chars))

In [31]:
# Random seed to help generate
start = numpy.random.randint(0, len(x_data) - 1)
pattern = x_data[start]
print("Random seed: ")
print("\"", ''.join([num_to_char[value] for value in pattern]), "\"")

Random seed: 
" intense labour wonderful discoveries modern philosophers always came studies discontented unsatisfie "


In [34]:
# generate the text
for i in range(100):
  x = numpy.reshape(pattern, (1, len(pattern), 1))
  x = x / float(vocab_len)
  prediction = model.predict(x, verbose=0)
  index = numpy.argmax(prediction)
  result = num_to_char[index]
  seq_in = [num_to_char[value] for value in pattern]
  sys.stdout.write(result)
  pattern.append(index)
  pattern = pattern[1:len(pattern)]

rs sears sears sears sears sears sears sears sears sears sears sears sears sears sears sears sears s

In [ ]:
# Result by the trained model: rs sears sears sears sears sears sears sears sears sears sears sears sears sears sears sears sears s
# Right now this model has been tarined only for 4 epochs , when trained fro longer it will give much more better results.